## Micrograd

In [55]:
import math

class Value:

    def __init__(self, data, _children = (), op = (), label = " "):
        self.data = data
        self.grad = 0
        self._backward = lambda: None 
        self._prev = set(_children)
        self._op = op
        self._label = label
    
    def __repr__(self):
        return f"Value(label={self._label}, data={self.data})"

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), "+")
        
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        
        out._backward = _backward 
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), "*")
        
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        
        out._backward = _backward 
        return out

    def tanh(self):
        n = self.data
        t = (math.exp(2*n) - 1)/(math.exp(2*n) + 1)
        out = Value(t, (self, ), label="tanh")

        def _backward():
            self.grad +=  (1 - t**2) * out.grad
        
        out._backward = _backward 
        return out

    def backprop(self):
        self._backward()
        print(f"label: {self._label}, grad: {self.grad}")
        for child in self._prev:
            child.backprop()

In [57]:
a = Value(3.0, label="a")
b = a + a
b.grad = 1
b.backprop()

label:  , grad: 1
label: a, grad: 2.0


In [58]:
x1 = Value(2.0, label="x1")
x2 = Value(0.0, label="x2")

w1 = Value(-3.0, label="w1")
w2 = Value(1.0, label="w2")

b = Value(6.8813735870195432, label="b")

x1w1 = x1*w1
x1w1.label = "x1*w1"

x2w2 = x2*w2
x2w2.label = "x2*w2"

x1w1x2w2 = x1w1 + x2w2
x1w1x2w2.label = "w1*w1 + x2*w2"

n = x1w1x2w2 + b
n.label = "n"

o = n.tanh()
o.label = "o"

In [59]:
o.grad = 1.0

In [60]:
o.backprop()

label: tanh, grad: 1.0
label:  , grad: 0.4999999999999999
label:  , grad: 0.4999999999999999
label:  , grad: 0.4999999999999999
label: w2, grad: 0.0
label: x2, grad: 0.4999999999999999
label:  , grad: 0.4999999999999999
label: w1, grad: 0.9999999999999998
label: x1, grad: -1.4999999999999996
label: b, grad: 0.4999999999999999
